In [1]:
from deck import full_euchre_deck
from dealer import Dealer
from n_play_round import round1, next_round
from tree_search import find_best_opener, trick_played, n_trick_sim
import numpy as np


In [2]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands5 = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands5

array([[[  0, -12],
        [  0, 100],
        [-10,   0],
        [-11,   0],
        [ 14,   0]],

       [[-12,   0],
        [  0, 140],
        [  9,   0],
        [  0, 110],
        [  0, 135]],

       [[ 11,   0],
        [  0, -10],
        [ -9,   0],
        [  0, 130],
        [  0, 120]],

       [[  0,  90],
        [ 10,   0],
        [  0, -13],
        [-13,   0],
        [  0,  -9]]])

In [3]:
best_opener = find_best_opener(hands=hands5, lead=0, tricks=5, previous_winners=np.array([]), sim_func=n_trick_sim)

[0.0256917  0.04022989 0.         0.         0.10280374]


In [4]:
best_opener

np.int64(4)

In [6]:
r2_leads, r2_hands = round1(hands_dealt=hands5, chosen_card=best_opener, leader=0)

In [7]:
# find the best response by iterating through each possible trick from the perspective of the opponent
# need to make this into a function as well

# team 1 (i.e. positions 1 and 3) should go for the best result
responder=1

r1_response_res = np.zeros(r2_leads.shape, dtype=np.float64)

for w in range(r2_leads.shape[0]):

    r3_leads, r3_hands, r2_score = next_round(
        current_hands=np.array([r2_hands[w]]),
        leads=np.array([r2_leads[w]]),
        game_round=2,
        game_score=np.array([r2_leads[w]]).reshape(-1, 1),
    )
    r4_leads, r4_hands, r3_score = next_round(
        current_hands=r3_hands, leads=r3_leads, game_round=3, game_score=r2_score
    )
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    # if eval_position % 2 == 0:
    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2)>=3

    r1_response_res[w] = np.mean(meta_results)

if responder % 2 == 0:
    best_response = np.argmax(r1_response_res)
else:
    best_response = np.argmin(r1_response_res)

print(best_response)

0


In [8]:
r1_response_res

array([0.89719626])

In [9]:
trick_played(hands5, r2_hands[best_response])

array([[14,  0],
       [ 9,  0],
       [11,  0],
       [10,  0]])

In [10]:
r1_winner = r2_leads[best_response]

In [11]:
r1_winner

np.int64(0)

In [12]:
r2_optimal =  r2_hands[best_response]

In [13]:
r2_optimal

array([[[  0, -12],
        [  0, 100],
        [-10,   0],
        [-11,   0]],

       [[-12,   0],
        [  0, 140],
        [  0, 110],
        [  0, 135]],

       [[  0, -10],
        [ -9,   0],
        [  0, 130],
        [  0, 120]],

       [[  0,  90],
        [  0, -13],
        [-13,   0],
        [  0,  -9]]])

In [14]:
r2_best_opener = find_best_opener(hands=r2_optimal, lead=r1_winner, tricks=4, previous_winners=np.array([r1_winner]), sim_func=n_trick_sim)

[0.         0.03703704 0.         0.        ]


In [16]:
r3_leads, r3_hands = round1(hands_dealt=r2_optimal, chosen_card=r2_best_opener, leader=r1_winner)

In [19]:
responder=1

r2_response_res = np.zeros(r3_leads.shape, dtype=np.float64)

for w in range(r3_leads.shape[0]):

    r4_leads, r4_hands, r3_score = next_round(
        current_hands=np.array([r3_hands[w]]),
        leads=np.array([r3_leads[w]]),
        game_round=3,
        game_score=np.array([r3_leads[w]]).reshape(-1, 1),
    )
    
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2 + (r1_winner%2))>=3

if responder % 2 == 0:
    best_response = np.argmin(r2_response_res)
else:
    best_response = np.argmax(r2_response_res)
    
print(best_response)

0


In [20]:
trick_played(r2_optimal, r3_hands[best_response])

array([[  0, 100],
       [  0, 140],
       [  0, 130],
       [  0,  90]])

In [21]:
r2_winner = r3_leads[best_response]

In [22]:
r3_optimal =  r3_hands[best_response]

In [23]:
r3_optimal

array([[[  0, -12],
        [-10,   0],
        [-11,   0]],

       [[-12,   0],
        [  0, 110],
        [  0, 135]],

       [[  0, -10],
        [ -9,   0],
        [  0, 120]],

       [[  0, -13],
        [-13,   0],
        [  0,  -9]]])

In [24]:
r3_best_opener = find_best_opener(hands=r3_optimal, lead=r2_winner, tricks=3, previous_winners=np.array([r1_winner, r2_winner]), sim_func=n_trick_sim)

[0. 0. 0.]


In [ ]:
r2_winner

np.int64(3)

In [ ]:
r4_leads, r4_hands = round1(hands_dealt=r3_optimal, chosen_card=r3_best_opener, leader=r2_winner)

In [ ]:
responder=1

r3_response_res = np.zeros(r4_leads.shape, dtype=np.float64)

for w in range(r4_leads.shape[0]):

    r5_leads, r5_hands, r4_score = next_round(
        current_hands=np.array([r4_hands[w]]),
        leads=np.array([r4_leads[w]]),
        game_round=4,
        game_score=np.array([r4_leads[w]]).reshape(-1, 1),
    )
    
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2 + (r1_winner%2)+(r2_winner%2))>=3

if responder % 2 == 0:
    best_response = np.argmin(r3_response_res)
else:
    best_response = np.argmax(r3_response_res)
    
print(best_response)

0


In [ ]:
trick_played(r3_optimal, r4_hands[best_response])

array([[ -9,   0],
       [  0, -12],
       [  0, -10],
       [  0, -13]])

In [ ]:
r3_winner = r4_leads[best_response]

In [ ]:
r3_winner

np.int64(3)

In [ ]:
(r1_winner % 2) + (r2_winner % 2) + (r3_winner % 2)

np.int64(3)

In [ ]:
# some general notes:

# Should probably move the functions in here to tree search and figure out the best way to refactor without having to repeat code so much